# Day 076 — Solution: Screen-Understanding Agent

In [ ]:
_SRC = '"""screen_agent.py — Day 076: Multimodal Screen-Understanding Agent.\n\nTwo-LLM pipeline: vision LLM (llava) describes the screen;\ntext LLM (llama3.2) reasons about a task in that visual context.\n\nFunctions:\n    capture_screenshot  — PIL.ImageGrab -> PIL Image\n    analyze_screenshot  — PIL Image + question -> str (llava)\n    describe_screen     — full scene description\n    read_screen_text    — verbatim text extraction\n    find_elements       — list UI elements of a given type\n    answer_about_screen — ad-hoc visual question\n    run_screen_task     — vision + text LLM -> {description, answer, task}\n    ScreenAgent         — stateful assistant, 3 injection points\n\nSetup:\n    pip install pillow ollama\n    ollama pull llava\n    ollama pull llama3.2\n"""\nimport io\nimport base64\nfrom pathlib import Path\n\n\ndef capture_screenshot(region=None, screenshot_fn=None):\n    """Capture a screenshot of the screen or a rectangular region.\n\n    Args:\n        region:        (left, top, right, bottom) pixel box, or None for full screen\n        screenshot_fn: callable(region) -> PIL.Image for testing\n    Returns:\n        PIL Image in RGB mode\n    """\n    if screenshot_fn is not None:\n        return screenshot_fn(region)\n    from PIL import ImageGrab\n    return ImageGrab.grab(bbox=region)\n\n\ndef analyze_screenshot(image, question, analyze_fn=None):\n    """Ask a vision LLM a question about an image.\n\n    Args:\n        image:      PIL Image\n        question:   natural-language question about the image\n        analyze_fn: callable(image, question) -> str for testing\n    Returns:\n        model answer string\n    """\n    if analyze_fn is not None:\n        return analyze_fn(image, question)\n    import ollama\n    buf = io.BytesIO()\n    image.save(buf, format=\'PNG\')\n    img_b64 = base64.b64encode(buf.getvalue()).decode()\n    resp = ollama.chat(\n        model=\'llava\',\n        messages=[{\'role\': \'user\', \'content\': question, \'images\': [img_b64]}],\n    )\n    return resp[\'message\'][\'content\']\n\n\ndef describe_screen(image, analyze_fn=None):\n    """Describe what is visible on screen."""\n    return analyze_screenshot(\n        image,\n        \'Describe what you see on this screen in detail.\',\n        analyze_fn=analyze_fn,\n    )\n\n\ndef read_screen_text(image, analyze_fn=None):\n    """Extract all visible text from the screen verbatim."""\n    return analyze_screenshot(\n        image,\n        \'Extract all visible text from this image exactly as it appears.\',\n        analyze_fn=analyze_fn,\n    )\n\n\ndef find_elements(image, element_type, analyze_fn=None):\n    """Find UI elements of a given type on screen."""\n    question = (\n        f\'List all {element_type} elements visible in this screenshot. \'\n        \'Be specific about their labels, text, or content.\'\n    )\n    return analyze_screenshot(image, question, analyze_fn=analyze_fn)\n\n\ndef answer_about_screen(image, question, analyze_fn=None):\n    """Answer an ad-hoc question about what is visible on screen."""\n    return analyze_screenshot(image, question, analyze_fn=analyze_fn)\n\n\ndef run_screen_task(image, task, analyze_fn=None, llm_fn=None):\n    """Analyze screenshot with vision LLM, then reason about task with text LLM.\n\n    Args:\n        image:      PIL Image (screenshot)\n        task:       task or question to answer using visual context\n        analyze_fn: callable(image, question) -> str (vision mock)\n        llm_fn:     callable(prompt) -> str (text LLM mock)\n    Returns:\n        dict with keys: description, answer, task\n    """\n    description = describe_screen(image, analyze_fn=analyze_fn)\n    lines = [\n        \'You are a screen-reading assistant.\',\n        \'Here is what is visible on screen:\',\n        \'\',\n        description,\n        \'\',\n        f\'Task: {task}\',\n        \'\',\n        \'Answer based only on what is visible on screen.\',\n    ]\n    context_prompt = \'\\n\'.join(lines)\n    if llm_fn is not None:\n        answer = llm_fn(context_prompt)\n    else:\n        import ollama\n        resp = ollama.chat(\n            model=\'llama3.2\',\n            messages=[{\'role\': \'user\', \'content\': context_prompt}],\n        )\n        answer = resp[\'message\'][\'content\']\n    return {\'description\': description, \'answer\': answer, \'task\': task}\n\n\nclass ScreenAgent:\n    """Stateful screen-understanding assistant.\n\n    Captures screenshots, analyzes them with a vision LLM, and reasons about\n    tasks using a text LLM. All three capabilities are injectable for testing.\n\n    Example::\n\n        from PIL import Image\n        agent = ScreenAgent(\n            screenshot_fn=lambda r: Image.new("RGB", (100, 100)),\n            analyze_fn=lambda img, q: "Mock description",\n            llm_fn=lambda p: "Mock answer",\n        )\n        img = agent.capture()\n        desc = agent.describe()\n        result = agent.run("What is the title of this window?")\n    """\n\n    def __init__(self, screenshot_fn=None, analyze_fn=None, llm_fn=None):\n        self._screenshot_fn = screenshot_fn\n        self._analyze_fn = analyze_fn\n        self._llm_fn = llm_fn\n        self._last_image = None\n        self._history = []\n\n    def capture(self, region=None):\n        """Capture a screenshot and store it as the current image."""\n        img = capture_screenshot(region=region, screenshot_fn=self._screenshot_fn)\n        self._last_image = img\n        self._history.append({\'action\': \'capture\', \'region\': region})\n        return img\n\n    def describe(self, image=None):\n        """Describe what is visible on screen."""\n        img = image if image is not None else self._last_image\n        if img is None:\n            raise ValueError(\'No image: call capture() first or pass image.\')\n        result = describe_screen(img, analyze_fn=self._analyze_fn)\n        self._history.append({\'action\': \'describe\', \'result\': result})\n        return result\n\n    def read_text(self, image=None):\n        """Extract text visible on screen."""\n        img = image if image is not None else self._last_image\n        if img is None:\n            raise ValueError(\'No image: call capture() first or pass image.\')\n        result = read_screen_text(img, analyze_fn=self._analyze_fn)\n        self._history.append({\'action\': \'read_text\', \'result\': result})\n        return result\n\n    def ask(self, question, image=None):\n        """Answer a question about what is visible on screen."""\n        img = image if image is not None else self._last_image\n        if img is None:\n            raise ValueError(\'No image: call capture() first or pass image.\')\n        result = answer_about_screen(img, question, analyze_fn=self._analyze_fn)\n        self._history.append({\'action\': \'ask\', \'question\': question, \'result\': result})\n        return result\n\n    def find(self, element_type, image=None):\n        """Find UI elements of a given type on screen."""\n        img = image if image is not None else self._last_image\n        if img is None:\n            raise ValueError(\'No image: call capture() first or pass image.\')\n        result = find_elements(img, element_type, analyze_fn=self._analyze_fn)\n        self._history.append({\'action\': \'find\', \'element_type\': element_type, \'result\': result})\n        return result\n\n    def run(self, task, image=None):\n        """Run a task against the current screenshot using vision + text reasoning."""\n        img = image if image is not None else self._last_image\n        if img is None:\n            img = self.capture()\n        result = run_screen_task(img, task, analyze_fn=self._analyze_fn, llm_fn=self._llm_fn)\n        self._history.append({\'action\': \'run\', \'task\': task, \'result\': result[\'answer\']})\n        return result\n\n    def history(self):\n        """Return a copy of the action history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the action history."""\n        self._history.clear()\n'
from pathlib import Path
Path('screen_agent.py').write_text(_SRC, encoding='utf-8')
print('screen_agent.py written.')

In [ ]:
import io, base64
from pathlib import Path
from PIL import Image as PILImage
from screen_agent import (
    capture_screenshot, analyze_screenshot, describe_screen,
    read_screen_text, find_elements, answer_about_screen,
    run_screen_task, ScreenAgent,
)

_mock_img = PILImage.new('RGB', (100, 100), color=(100, 100, 100))
_mock_screenshot_fn = lambda region=None: _mock_img
_mock_analyze_fn    = lambda img, q: 'SCREEN:' + q[:12]
_mock_llm_fn        = lambda p: 'ANSWER:' + p[:8]

# 1. capture_screenshot
img = capture_screenshot(screenshot_fn=_mock_screenshot_fn)
assert isinstance(img, PILImage.Image)
print("\u2705 capture_screenshot correct")

# 2. analyze_screenshot
r = analyze_screenshot(_mock_img, 'Q?', analyze_fn=_mock_analyze_fn)
assert isinstance(r, str)
print("\u2705 analyze_screenshot correct")

# 3. describe_screen
d = describe_screen(_mock_img, analyze_fn=_mock_analyze_fn)
assert isinstance(d, str)
print("\u2705 describe_screen correct")

# 4. read_screen_text
t = read_screen_text(_mock_img, analyze_fn=_mock_analyze_fn)
assert isinstance(t, str)
print("\u2705 read_screen_text correct")

# 5. find_elements
qs = []
find_elements(_mock_img, 'button', analyze_fn=lambda i, q: (qs.append(q), 'F')[1])
assert 'button' in qs[0]
print("\u2705 find_elements correct")

# 6. answer_about_screen
a = answer_about_screen(_mock_img, 'Count?', analyze_fn=_mock_analyze_fn)
assert isinstance(a, str)
print("\u2705 answer_about_screen correct")

# 7. run_screen_task
res = run_screen_task(_mock_img, 'Find title',
                      analyze_fn=_mock_analyze_fn, llm_fn=_mock_llm_fn)
assert isinstance(res, dict) and all(k in res for k in ('description', 'answer', 'task'))
print("\u2705 run_screen_task correct")

# 8. ScreenAgent
agent = ScreenAgent(screenshot_fn=_mock_screenshot_fn,
                    analyze_fn=_mock_analyze_fn,
                    llm_fn=_mock_llm_fn)
agent.capture()
agent.describe()
agent.ask('Q?')
agent.find('button')
result = agent.run('Find the title')
assert isinstance(result, dict) and 'answer' in result
hist = agent.history()
assert len(hist) == 5
agent.clear_history()
assert agent.history() == []
print("\u2705 ScreenAgent correct")
print("\nMultimodal Agent complete!")
